In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
df = pd.read_excel("2. Iris Dataset.xlsx")

df.head()

,Sepal Length,Sepal Width,Petal Length,Petal Width,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (73, 5)

Columns:
['Sepal Length', 'Sepal Width', 'Petal Length', 'Petal Width', 'Species']

Missing values:
Sepal Length    0
Sepal Width     0
Petal Length    0
Petal Width     0
Species         0
dtype: int64


In [4]:
X = df[
    ["Sepal Length", "Sepal Width", "Petal Length", "Petal Width"]
]

y = df["Species"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (73, 4)
Target shape: (73,)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model accuracy:", accuracy)

Model accuracy: 1.0


In [6]:
joblib.dump(model, "iris_model.joblib")

print("Model saved successfully.")

Model saved successfully.


In [7]:
%pip install flask


   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   -------- ------------------------------- 1/5 [itsdangerous]
   ---------------- ----------------------- 2/5 [click]
   ---------------- ----------------------- 2/5 [click]
   ---------------- ----------------------- 2/5 [click]
   ---------------- ----------------------- 2/5 [click]
   -------------------------------- ------- 4/5 [flask]
   -------------------------------- ------- 4/5 [flask]
   -------------------------------- ------- 4/5 [flask]
   ----------

In [8]:
import flask

print("Flask version:", flask.__version__)

Flask version: 3.1.3


C:\Users\M K SHUKLA\AppData\Local\Temp\ipykernel_12220\2642122357.py:3: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version:", flask.__version__)


In [9]:
api_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

model = joblib.load("iris_model.joblib")


@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Iris classification API is running"
    })


@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        required_features = [
            "Sepal Length",
            "Sepal Width",
            "Petal Length",
            "Petal Width"
        ]

        for feature in required_features:
            if feature not in data:
                return jsonify({
                    "error": f"Missing feature: {feature}"
                }), 400

        features = np.array([[
            data["Sepal Length"],
            data["Sepal Width"],
            data["Petal Length"],
            data["Petal Width"]
        ]])

        prediction = model.predict(features)[0]

        return jsonify({
            "prediction": str(prediction)
        })

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500


if __name__ == "__main__":
    app.run(debug=True)
'''

with open("app.py", "w") as file:
    file.write(api_code)

print("app.py created successfully.")

app.py created successfully.


In [10]:
import os

print("app.py exists:", os.path.exists("app.py"))
print("iris_model.joblib exists:", os.path.exists("iris_model.joblib"))

app.py exists: True
iris_model.joblib exists: True


In [11]:
requirements = """flask
joblib
numpy
pandas
scikit-learn
openpyxl
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully.")

requirements.txt created successfully.


In [12]:
import py_compile

py_compile.compile(
    "app.py",
    doraise=True
)

print("app.py syntax is correct.")

app.py syntax is correct.


In [13]:
api_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np
import os

app = Flask(__name__)

model_path = os.path.join(
    os.path.dirname(os.path.abspath(__file__)),
    "iris_model.joblib"
)

model = joblib.load(model_path)


@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Iris classification API is running"
    })


@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        if not data:
            return jsonify({
                "error": "Request body must contain JSON data."
            }), 400

        required_features = [
            "Sepal Length",
            "Sepal Width",
            "Petal Length",
            "Petal Width"
        ]

        for feature in required_features:
            if feature not in data:
                return jsonify({
                    "error": f"Missing feature: {feature}"
                }), 400

        try:
            features = np.array([[
                float(data["Sepal Length"]),
                float(data["Sepal Width"]),
                float(data["Petal Length"]),
                float(data["Petal Width"])
            ]])
        except (ValueError, TypeError):
            return jsonify({
                "error": "All feature values must be numeric."
            }), 400

        prediction = model.predict(features)[0]

        return jsonify({
            "prediction": str(prediction)
        }), 200

    except Exception as e:
        return jsonify({
            "error": "Model prediction failed.",
            "details": str(e)
        }), 500


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False)
'''

with open("app.py", "w") as file:
    file.write(api_code)

print("Final app.py created successfully.")

Final app.py created successfully.


In [14]:
import py_compile

py_compile.compile(
    "app.py",
    doraise=True
)

print("app.py syntax is correct.")

app.py syntax is correct.


In [15]:
import subprocess
import sys
import time

server = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(3)

print("Flask API started.")
print("Process running:", server.poll() is None)

Flask API started.
Process running: True


In [16]:
import urllib.request

response = urllib.request.urlopen(
    "http://127.0.0.1:5000/"
)

print(response.read().decode())

{"message":"Iris classification API is running"}



In [17]:
import json
import urllib.request

data = {
    "Sepal Length": 5.1,
    "Sepal Width": 3.5,
    "Petal Length": 1.4,
    "Petal Width": 0.2
}

request_data = json.dumps(data).encode("utf-8")

request = urllib.request.Request(
    "http://127.0.0.1:5000/predict",
    data=request_data,
    headers={
        "Content-Type": "application/json"
    },
    method="POST"
)

response = urllib.request.urlopen(request)

print(response.read().decode())

{"prediction":"Iris-setosa"}



In [18]:
server.terminate()
server.wait()

print("Flask API stopped.")

Flask API stopped.


In [19]:
%pip install pytest


   ---------------------------------------- 0/3 [pluggy]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   ---------------------------------------- 3/3 [pytest]

Note: you may need to restart

In [20]:
import os

os.makedirs("tests", exist_ok=True)

print("tests folder created.")

tests folder created.


In [21]:
test_code = '''
import sys
import os

sys.path.insert(
    0,
    os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
)

from app import app


def test_home_endpoint():
    client = app.test_client()

    response = client.get("/")

    assert response.status_code == 200

    data = response.get_json()

    assert data["message"] == "Iris classification API is running"


def test_predict_endpoint():
    client = app.test_client()

    data = {
        "Sepal Length": 5.1,
        "Sepal Width": 3.5,
        "Petal Length": 1.4,
        "Petal Width": 0.2
    }

    response = client.post(
        "/predict",
        json=data
    )

    assert response.status_code == 200

    result = response.get_json()

    assert "prediction" in result


def test_missing_feature():
    client = app.test_client()

    data = {
        "Sepal Length": 5.1,
        "Sepal Width": 3.5,
        "Petal Length": 1.4
    }

    response = client.post(
        "/predict",
        json=data
    )

    assert response.status_code == 400

    result = response.get_json()

    assert "error" in result


def test_invalid_feature_type():
    client = app.test_client()

    data = {
        "Sepal Length": "invalid",
        "Sepal Width": 3.5,
        "Petal Length": 1.4,
        "Petal Width": 0.2
    }

    response = client.post(
        "/predict",
        json=data
    )

    assert response.status_code == 400

    result = response.get_json()

    assert "error" in result
'''

with open("tests/test_app.py", "w") as file:
    file.write(test_code)

print("test_app.py created successfully.")

test_app.py created successfully.


In [22]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-v"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\M K SHUKLA\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\M K SHUKLA
plugins: anyio-4.15.1
collecting ... collected 4 items

tests/test_app.py::test_home_endpoint PASSED                             [ 25%]
tests/test_app.py::test_predict_endpoint PASSED                          [ 50%]
tests/test_app.py::test_missing_feature PASSED                           [ 75%]
tests/test_app.py::test_invalid_feature_type PASSED                      [100%]

============================== warnings summary ===============================
tests/test_app.py::test_predict_endpoint
  C:\Users\M K SHUKLA\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
    warnings.

In [23]:
import os

os.makedirs(".github/workflows", exist_ok=True)

print("GitHub Actions folder created.")

GitHub Actions folder created.


In [24]:
workflow = """
name: Iris Model API CI

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run tests
        run: |
          pytest tests -v
"""

with open(".github/workflows/ci.yml", "w") as file:
    file.write(workflow)

print("GitHub Actions CI pipeline created successfully.")

GitHub Actions CI pipeline created successfully.


In [25]:
print(open(".github/workflows/ci.yml").read())


name: Iris Model API CI

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run tests
        run: |
          pytest tests -v



In [26]:
api_documentation = """
# Iris Classification REST API

## Overview

This project provides a REST API for predicting the species of an Iris flower using a trained Random Forest classification model.

The API is built using Flask.

## Running the API

Install the required packages:

pip install -r requirements.txt

Start the API:

python app.py

The API runs at:

http://127.0.0.1:5000

## Home Endpoint

### Request

GET /

### Response

{
    "message": "Iris classification API is running"
}

## Prediction Endpoint

### Request

POST /predict

The request must contain four numerical features:

- Sepal Length
- Sepal Width
- Petal Length
- Petal Width

### Sample Request

{
    "Sepal Length": 5.1,
    "Sepal Width": 3.5,
    "Petal Length": 1.4,
    "Petal Width": 0.2
}

### Sample Response

{
    "prediction": "setosa"
}

## Error Handling

If a required feature is missing, the API returns HTTP status code 400.

Example:

{
    "error": "Missing feature: Petal Width"
}

If a feature contains a non-numeric value, the API returns HTTP status code 400.

Example:

{
    "error": "All feature values must be numeric."
}

If an unexpected model prediction error occurs, the API returns HTTP status code 500.

## Testing

Run the tests using:

pytest tests -v
"""

with open("API_DOCUMENTATION.md", "w") as file:
    file.write(api_documentation)

print("API_DOCUMENTATION.md created successfully.")

API_DOCUMENTATION.md created successfully.


In [27]:
from app import app

client = app.test_client()

print("Flask test client created.")

Flask test client created.


In [28]:
response = client.post(
    "/predict",
    json={
        "Sepal Length": 5.1,
        "Sepal Width": 3.5,
        "Petal Length": 1.4
    }
)

print("Status code:", response.status_code)
print("Response:", response.get_json())

Status code: 400
Response: {'error': 'Missing feature: Petal Width'}


In [30]:
response = client.post(
    "/predict",
    json={
        "Sepal Length": "wrong",
        "Sepal Width": 3.5,
        "Petal Length": 1.4,
        "Petal Width": 0.2
    }
)

print("Status code:", response.status_code)
print("Response:", response.get_json())

Status code: 400
Response: {'error': 'All feature values must be numeric.'}


In [31]:
import os

required_files = [
    "2. Iris Dataset.xlsx",
    "iris_model.joblib",
    "app.py",
    "requirements.txt",
    "API_DOCUMENTATION.md",
    "tests/test_app.py",
    ".github/workflows/ci.yml"
]

for file in required_files:
    print(f"{file}: {os.path.exists(file)}")

2. Iris Dataset.xlsx: True
iris_model.joblib: True
app.py: True
requirements.txt: True
API_DOCUMENTATION.md: True
tests/test_app.py: True
.github/workflows/ci.yml: True


In [32]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-v"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\M K SHUKLA\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\M K SHUKLA
plugins: anyio-4.15.1
collecting ... collected 4 items

tests/test_app.py::test_home_endpoint PASSED                             [ 25%]
tests/test_app.py::test_predict_endpoint PASSED                          [ 50%]
tests/test_app.py::test_missing_feature PASSED                           [ 75%]
tests/test_app.py::test_invalid_feature_type PASSED                      [100%]

============================== warnings summary ===============================
tests/test_app.py::test_predict_endpoint
  C:\Users\M K SHUKLA\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
    warnings.

In [33]:
server = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

import time
time.sleep(3)

print("API running:", server.poll() is None)

API running: True


In [34]:
import urllib.request

response = urllib.request.urlopen(
    "http://127.0.0.1:5000/"
)

print(response.read().decode())

{"message":"Iris classification API is running"}



In [35]:
import json
import urllib.request

data = {
    "Sepal Length": 5.1,
    "Sepal Width": 3.5,
    "Petal Length": 1.4,
    "Petal Width": 0.2
}

request_data = json.dumps(data).encode("utf-8")

request = urllib.request.Request(
    "http://127.0.0.1:5000/predict",
    data=request_data,
    headers={
        "Content-Type": "application/json"
    },
    method="POST"
)

response = urllib.request.urlopen(request)

print(response.read().decode())

{"prediction":"Iris-setosa"}



In [36]:
server.terminate()
server.wait()

print("API stopped.")

API stopped.


In [37]:
git_path = r"C:\Program Files\Git\cmd\git.exe"

print("Git exists:", os.path.exists(git_path))

Git exists: True


In [38]:
subprocess.run(
    [git_path, "config", "user.name", "Aaryan Shukla"]
)

subprocess.run(
    [git_path, "config", "user.email", "aaryanshukla033@gmail.com"]
)

print("Git identity configured.")

Git identity configured.


In [39]:
result = subprocess.run(
    [git_path, "add", "."],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


error: open("AppData/Local/Comms/UnistoreDB/USS.jtx"): Permission denied
error: unable to index file 'AppData/Local/Comms/UnistoreDB/USS.jtx'
fatal: adding files failed



In [40]:
result = subprocess.run(
    [git_path, "add", "."],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


error: open("AppData/Local/Comms/UnistoreDB/USS.jtx"): Permission denied
error: unable to index file 'AppData/Local/Comms/UnistoreDB/USS.jtx'
fatal: adding files failed



In [42]:
import os

base_folder = os.getcwd()

print("Current folder:")
print(base_folder)

print("\nSearching for the Assignment 2 project folder...")

project_folder = None

for root, dirs, files in os.walk(base_folder):
    if "app.py" in files and "iris_model.joblib" in files:
        project_folder = root
        break

print("\nProject folder found:")
print(project_folder)

Current folder:
C:\Users\M K SHUKLA

Searching for the Assignment 2 project folder...

Project folder found:
C:\Users\M K SHUKLA


In [43]:
import os
import shutil

home_folder = r"C:\Users\M K SHUKLA"

project_folder = os.path.join(
    home_folder,
    "Model_Deployment_Assignment_2"
)

os.makedirs(project_folder, exist_ok=True)

files_to_copy = [
    "2. Iris Dataset.xlsx",
    "iris_model.joblib",
    "app.py",
    "requirements.txt",
    "API_DOCUMENTATION.md"
]

for file in files_to_copy:
    source = os.path.join(home_folder, file)
    
    if os.path.exists(source):
        shutil.copy2(
            source,
            os.path.join(project_folder, file)
        )
        print("Copied:", file)
    else:
        print("NOT FOUND:", file)

folders_to_copy = [
    "tests",
    ".github"
]

for folder in folders_to_copy:
    source = os.path.join(home_folder, folder)
    
    if os.path.exists(source):
        destination = os.path.join(project_folder, folder)
        
        if os.path.exists(destination):
            shutil.rmtree(destination)
        
        shutil.copytree(source, destination)
        print("Copied folder:", folder)

print("\nProject folder created:")
print(project_folder)

Copied: 2. Iris Dataset.xlsx
Copied: iris_model.joblib
Copied: app.py
Copied: requirements.txt
Copied: API_DOCUMENTATION.md
Copied folder: tests
Copied folder: .github

Project folder created:
C:\Users\M K SHUKLA\Model_Deployment_Assignment_2


In [44]:
print("Files in Assignment 2 folder:\n")

for root, dirs, files in os.walk(project_folder):
    level = root.replace(project_folder, "").count(os.sep)
    indent = "    " * level
    
    print(f"{indent}{os.path.basename(root)}/")
    
    for file in files:
        print(f"{indent}    {file}")

Files in Assignment 2 folder:

Model_Deployment_Assignment_2/
    2. Iris Dataset.xlsx
    API_DOCUMENTATION.md
    app.py
    iris_model.joblib
    requirements.txt
    .github/
        workflows/
            ci.yml
    tests/
        test_app.py
        __pycache__/
            test_app.cpython-313-pytest-9.1.1.pyc


In [45]:
result = subprocess.run(
    [git_path, "init", "-b", "main"],
    cwd=project_folder,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Initialized empty Git repository in C:/Users/M K SHUKLA/Model_Deployment_Assignment_2/.git/




In [46]:
subprocess.run(
    [git_path, "config", "user.name", "Aaryan Shukla"],
    cwd=project_folder
)

subprocess.run(
    [git_path, "config", "user.email", "aaryanshukla033@gmail.com"],
    cwd=project_folder
)

print("Git identity configured.")

Git identity configured.


In [47]:
result = subprocess.run(
    [git_path, "status"],
    cwd=project_folder,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.github/
	2. Iris Dataset.xlsx
	API_DOCUMENTATION.md
	app.py
	iris_model.joblib
	requirements.txt
	tests/

nothing added to commit but untracked files present (use "git add" to track)




In [49]:
result = subprocess.run(
    [git_path, "add", "."],
    cwd=project_folder,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

In [50]:
result = subprocess.run(
    [git_path, "status"],
    cwd=project_folder,
    capture_output=True,
    text=True
)

print(result.stdout)

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .github/workflows/ci.yml
	new file:   2. Iris Dataset.xlsx
	new file:   API_DOCUMENTATION.md
	new file:   app.py
	new file:   iris_model.joblib
	new file:   requirements.txt
	new file:   tests/__pycache__/test_app.cpython-313-pytest-9.1.1.pyc
	new file:   tests/test_app.py




In [51]:
result = subprocess.run(
    [
        git_path,
        "commit",
        "-m",
        "Complete Assignment 2 REST API and CI"
    ],
    cwd=project_folder,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

[main (root-commit) bf2ac21] Complete Assignment 2 REST API and CI
 8 files changed, 282 insertions(+)
 create mode 100644 .github/workflows/ci.yml
 create mode 100644 2. Iris Dataset.xlsx
 create mode 100644 API_DOCUMENTATION.md
 create mode 100644 app.py
 create mode 100644 iris_model.joblib
 create mode 100644 requirements.txt
 create mode 100644 tests/__pycache__/test_app.cpython-313-pytest-9.1.1.pyc
 create mode 100644 tests/test_app.py




In [52]:
result = subprocess.run(
    [git_path, "status"],
    cwd=project_folder,
    capture_output=True,
    text=True
)

print(result.stdout)

On branch main
nothing to commit, working tree clean

